# mini-vLLM — GPU Correctness & Benchmark Notebook

**Runtime:** GPU T4 — Kaggle: Settings → Accelerator → GPU T4 x1

In [ ]:
# ── Clone / update ────────────────────────────────────────────────────────
import os
from pathlib import Path

BRANCH    = 'milestone/m1-correctness-baseline'
CLONE_URL = 'https://github.com/shlokkvaishnav/LLM-Inference-Engine.git'
LOCAL_DIR = 'mini-vllm'

if Path(LOCAL_DIR).exists():
    !git -C {LOCAL_DIR} pull
else:
    !git clone --branch {BRANCH} {CLONE_URL} {LOCAL_DIR}

os.chdir(LOCAL_DIR)
print('Working dir:', os.getcwd())

In [ ]:
# Install our package without touching torch/torchvision.
# Kaggle ships a compatible PyTorch build — reinstalling torch would break
# the torchvision ABI (register_fake conflict). We use --no-deps to skip
# torch, then install only what Kaggle doesn't pre-ship.
!pip install -e . --no-deps -q
!pip install transformers tokenizers accelerate pydantic fastapi "uvicorn[standard]" httpx tqdm pytest pytest-asyncio -q
print('Install complete.')

In [ ]:
import torch
print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU     : {torch.cuda.get_device_name(0)}')
    print(f'VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## M1 + M2 — Correctness tests

- **M1:** our decode loop matches `model.generate()` token-for-token (greedy, single sequence)
- **M2:** every sequence in a static batch matches its single-sequence result (left-padding correctness)

In [ ]:
!MINI_VLLM_TEST_MODEL='TinyLlama/TinyLlama-1.1B-Chat-v1.0' \
 MINI_VLLM_TEST_DEVICE='cuda' \
 python -m pytest tests/test_correctness.py -v -s 2>&1

---
## M3 — Continuous batching *(added when milestone lands)*
## M4 — Paged KV-cache *(added when milestone lands)*
## M5 — Quantization *(added when milestone lands)*
## M7 — Benchmark suite + P50/P95/P99 load test *(added when milestone lands)*